# MRKR Dataset Download
**Emory Knee Radiograph Dataset → Google Drive**

Run every cell top to bottom. The notebook will:
1. Ask you to upload your credentials JSON (one click)
2. Download the metadata index (210 MB)
3. Filter to thesis-relevant images and measure real file sizes
4. Show a download plan before touching your Drive quota
5. Download in parallel batches with full resume support
6. Verify the result

**Drive layout after completion:**
```
Master Thesis/MRKR/
    dicoms/          ← downloaded DICOMs (~700–920 GB)
    metadata/        ← CSV index files
    download_log/    ← progress log + error log
```

## 0 — Mount Drive & Install Packages

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       '--quiet', 'boto3', 'tqdm'])
print('Packages ready.')

Packages ready.


---
## 1 — Upload Credentials JSON
Click **Choose Files** below and select the JSON file you received from
[data.hitilab.com](https://data.hitilab.com). The file stays in Colab memory only —
it is never written to Drive.

In [3]:
import json
from google.colab import files as colab_files
from pathlib import Path

print('Waiting for file upload...')
uploaded = colab_files.upload()

if not uploaded:
    raise RuntimeError('No file selected. Re-run this cell.')

fname = list(uploaded.keys())[0]
creds = json.loads(uploaded[fname].decode('utf-8'))
print(f'Loaded: {fname}')
print(f'Keys  : {list(creds.keys())}')

def _get(d, *keys):
    for k in keys:
        if k in d: return d[k]
    return None

AWS_ACCESS_KEY_ID     = _get(creds,
    'AccessKeyId',       'access_key_id',
    'aws_access_key_id', 'access_key')
AWS_SECRET_ACCESS_KEY = _get(creds,
    'SecretAccessKey',   'secret_access_key',
    'aws_secret_access_key', 'secret_key')
AWS_SESSION_TOKEN     = _get(creds,
    'SessionToken',      'session_token',
    'aws_session_token')

S3_BUCKET = _get(creds, 'bucket', 's3_bucket') or 'emory-mrkr-dataset'

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print('Keys found in JSON:', list(creds.keys()))
    raise ValueError('Credentials not found. Contact bprice9@emory.edu')

print()
print(f'Access Key : {AWS_ACCESS_KEY_ID[:6]}...{AWS_ACCESS_KEY_ID[-4:]}')
print(f'S3 Bucket  : {S3_BUCKET}')
print(f'Session    : {"present" if AWS_SESSION_TOKEN else "not present"}')
print('Credentials ready.')


Waiting for file upload...


Saving aws_s3.json to aws_s3.json
Loaded: aws_s3.json
Keys  : ['aws_access_key_id', 'aws_secret_access_key', 'bucket']

Access Key : AKIAZE...54MN
S3 Bucket  : emory-mrkr-dataset
Session    : not present
Credentials ready.


---
## 2 — Configuration & Paths

In [4]:
from pathlib import Path

AWS_REGION = 'us-west-2'

DRIVE_BASE = Path('/content/drive/MyDrive/Master Thesis/MRKR')
DICOM_DIR  = DRIVE_BASE / 'dicoms'
META_DIR   = DRIVE_BASE / 'metadata'
LOG_DIR    = DRIVE_BASE / 'download_log'

for d in [DICOM_DIR, META_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BUDGET_GB = 1_500
MAX_WORKERS     = 8

FILTER_VIEW_POSITION   = 'F'
FILTER_BILATERAL       = True
FILTER_NO_ARTHROPLASTY = True
FILTER_REQUIRE_KL      = True

print('=== Drive layout ===')
print(f'  {DRIVE_BASE}/')
print(f'    dicoms/         <- DICOMs go here')
print(f'    metadata/       <- CSV index files')
print(f'    download_log/   <- progress + error logs')
print(f'  Budget  : {DRIVE_BUDGET_GB} GB  (200 GB safety margin from 1.7 TB)')
print(f'  Workers : {MAX_WORKERS} parallel threads')


=== Drive layout ===
  /content/drive/MyDrive/Master Thesis/MRKR/
    dicoms/         <- DICOMs go here
    metadata/       <- CSV index files
    download_log/   <- progress + error logs
  Budget  : 1500 GB  (200 GB safety margin from 1.7 TB)
  Workers : 8 parallel threads


---
## 3 — Connect to S3

In [5]:
import boto3
from botocore.config import Config

kwargs = dict(
    aws_access_key_id     = AWS_ACCESS_KEY_ID,
    aws_secret_access_key = AWS_SECRET_ACCESS_KEY,
    region_name           = AWS_REGION,
    config                = Config(
        retries              = {'max_attempts': 5, 'mode': 'adaptive'},
        max_pool_connections = MAX_WORKERS + 4,
    )
)
if AWS_SESSION_TOKEN:
    kwargs['aws_session_token'] = AWS_SESSION_TOKEN

s3 = boto3.client('s3', **kwargs)

try:
    resp = s3.list_objects_v2(Bucket=S3_BUCKET, MaxKeys=5)
    print(f'Connected to  s3://{S3_BUCKET}')
    print('Sample keys:')
    for obj in resp.get('Contents', []):
        print(f'  {obj["Key"][:90]}')
except Exception as e:
    print(f'ERROR: {e}')
    print()
    print('If you see "Token has expired" re-run Cell 1 with a fresh credentials file.')
    print('hitilab.com credentials expire after a set time period.')

Connected to  s3://emory-mrkr-dataset
Sample keys:
  images/10001182/1.2.849.113972.3.57.1.62631357.20201023.1122916/1.2.840.113570.16324617021
  images/10001182/1.2.849.113972.3.57.1.62631357.20201023.1122916/1.2.845.113566.16324617021
  images/10001182/1.2.849.113972.3.57.1.62631357.20201023.1122916/1.2.847.113572.16324617022
  images/10001758/1.2.840.113975.3.59.1.64324621.20210625.1133227/1.2.842.113566.16324621223
  images/10001758/1.2.840.113975.3.59.1.64324621.20210625.1133227/1.2.842.113572.16324621223


---
## 4 — Download Metadata CSV
Downloads `MRKR_image_metadata.csv` (210 MB) — the index of all 503,261 images.
All filtering is done from this file so no DICOMs are wasted.

In [6]:
import time

META_KEY = 'tables/MRKR_image_metadata.csv'
DEMO_KEY = 'tables/MRKR_demographics.csv'

META_CSV = META_DIR / 'MRKR_image_metadata.csv'
DEMO_CSV = META_DIR / 'MRKR_demographics.csv'

if META_CSV.exists() and META_CSV.stat().st_size > 1_000_000:
    print(f'Metadata already on Drive ({META_CSV.stat().st_size/1e6:.0f} MB) -- skipping.')
else:
    print(f'Downloading {META_KEY} (~219 MB) ...')
    t0       = time.time()
    resp     = s3.get_object(Bucket=S3_BUCKET, Key=META_KEY)
    total    = int(resp['ContentLength'])
    chunk_sz = 8 * 1024 * 1024
    done     = 0
    with open(META_CSV, 'wb') as fh:
        for chunk in resp['Body'].iter_chunks(chunk_size=chunk_sz):
            fh.write(chunk)
            done += len(chunk)
            pct   = 100 * done / total
            print(f'  {done/1e6:.0f} / {total/1e6:.0f} MB  ({pct:.1f}%)', end='\r')
    print(f'\nDone in {time.time()-t0:.0f}s')

if not DEMO_CSV.exists():
    print(f'Downloading {DEMO_KEY} (4.6 MB) ...')
    s3.download_file(S3_BUCKET, DEMO_KEY, str(DEMO_CSV))
    print(f'Saved: {DEMO_CSV.name}')

print()
print('Metadata ready.')
print(f'  {META_CSV}')
print(f'  {DEMO_CSV}')


Metadata already on Drive (219 MB) -- skipping.

Metadata ready.
  /content/drive/MyDrive/Master Thesis/MRKR/metadata/MRKR_image_metadata.csv
  /content/drive/MyDrive/Master Thesis/MRKR/metadata/MRKR_demographics.csv


---
## 5 — Filter & Measure
Applies thesis filters, measures real DICOM sizes from a 100-file sample,
then builds the final stratified download plan.

In [7]:
import pandas as pd
import numpy as np

print('Loading metadata CSV ...')
meta = pd.read_csv(str(META_CSV), low_memory=False)
print(f'Total rows : {len(meta):,}')
print()

for col in ['arthroplasty', 'laterality', 'view_position']:
    vc = meta[col].value_counts(dropna=False).head(8)
    print(f'{col} value counts:')
    for val, n in vc.items():
        print(f'  {repr(val):30s}  {n:>8,}')
    print()

pool = meta.copy()
print(f'Start                                    : {len(pool):>8,}')

pool = pool[pool['view_position'] == 'F']
print(f'Frontal (view_position=F)                : {len(pool):>8,}')

ARTHROPLASTY_VALUES = {'R', 'L', 'B', 'NL', 'r', 'l', 'b', 'nl'}
pool = pool[~pool['arthroplasty'].astype(str).str.strip().str.upper().isin(
    {'R', 'L', 'B', 'NL'})]
print(f'No arthroplasty                          : {len(pool):>8,}')

lat_vals = set(pool['laterality'].dropna().unique())

bilateral_markers = lat_vals & {'B', 'b', 'bilateral', 'Bilateral'}
if bilateral_markers:
    pool = pool[pool['laterality'].isin(bilateral_markers)]
else:

    print(f'  NOTE: No bilateral marker found in: {lat_vals}')
    print(f'  Keeping all lateralities — will crop R and L separately')
print(f'After laterality filter                  : {len(pool):>8,}')

pool = pool[
    pool['R_KLG_inference'].notna() |
    pool['L_KLG_inference'].notna()
]
print(f'Has KL grade                             : {len(pool):>8,}')

pool = pool.dropna(subset=['dicom_path']).reset_index(drop=True)
print(f'Valid dicom_path                         : {len(pool):>8,}')
print()

sample = pool['dicom_path'].dropna().head(3).tolist()
print('Sample dicom_path (CSV value):')
for p in sample:
    print(f'  {p}')
print('S3 key will be: images/<dicom_path>')
print()

if len(pool) > 0:
    kl_r = pd.to_numeric(pool['R_KLG_inference'], errors='coerce')
    kl_l = pd.to_numeric(pool['L_KLG_inference'], errors='coerce')
    all_kl = pd.concat([kl_r, kl_l]).dropna().astype(int)
    print('KL grade distribution:')
    vc = all_kl.value_counts().sort_index()
    for k, n in vc.items():
        bar = '#' * int(n / len(all_kl) * 35)
        print(f'  KL{k} : {n:>7,}  ({100*n/len(all_kl):.1f}%)  {bar}')
    print()
    print(f'Films in pool            : {len(pool):,}')
    print(f'Individual knee crops    : ~{len(pool)*2:,}  (split each bilateral)')
else:
    print('WARNING: pool is empty. Check the value counts printed above.')
    print('Adjust filter values to match what is actually in your CSV.')


Loading metadata CSV ...
Total rows : 503,261

arthroplasty value counts:
  '0'                              388,648
  'R'                               55,579
  'L'                               50,023
  'B'                                8,982
  'NL'                                  29

laterality value counts:
  'R'                              184,327
  'L'                              176,945
  'B'                              141,307
  '-1'                                 682

view_position value counts:
  'F'                              205,121
  'L'                              195,802
  'S'                               70,250
  'I'                               27,177
  'E'                                4,911

Start                                    :  503,261
Frontal (view_position=F)                :  205,121
No arthroplasty                          :  154,696
After laterality filter                  :   81,271
Has KL grade                             :   81,104
Valid di

In [8]:
import numpy as np

if len(pool) == 0:
    raise RuntimeError('Pool is empty — fix the filters in the previous cell first.')

print('Measuring real DICOM sizes (100-file sample) ...')
sample_paths = pool['dicom_path'].sample(min(100, len(pool)), random_state=42).tolist()

S3_IMG_PREFIX = 'images/'
sizes = []
for path in sample_paths:
    key = S3_IMG_PREFIX + str(path) if not str(path).startswith('images/') else str(path)
    try:
        r = s3.head_object(Bucket=S3_BUCKET, Key=key)
        sizes.append(r['ContentLength'])
    except Exception:
        pass

avg_mb = np.mean(sizes) / 1e6 if sizes else 5.0
p25_mb = np.percentile(sizes, 25) / 1e6 if sizes else avg_mb
p75_mb = np.percentile(sizes, 75) / 1e6 if sizes else avg_mb

print(f'  Measured from {len(sizes)} files:')
print(f'  Average : {avg_mb:.1f} MB')
print(f'  P25-P75 : {p25_mb:.1f} MB - {p75_mb:.1f} MB')
print()

max_dicoms = int(DRIVE_BUDGET_GB * 1024 / avg_mb)

pool2 = pool.copy()
pool2['kl_max'] = (
    pd.to_numeric(pool2['R_KLG_inference'], errors='coerce')
    .fillna(pd.to_numeric(pool2['L_KLG_inference'], errors='coerce'))
    .fillna(0).astype(int).clip(0, 4)
)

total_pool_gb = len(pool2) * avg_mb / 1024
print(f'Full filtered pool  : {len(pool2):,} DICOMs  (~{total_pool_gb:.0f} GB)')
print(f'Drive budget        : {DRIVE_BUDGET_GB} GB')
print(f'Max DICOMs in budget: {max_dicoms:,}')
print()

if max_dicoms >= len(pool2):
    plan = pool2.copy()
    print(f'Entire pool fits -- downloading ALL {len(plan):,} DICOMs')
else:
    per_grade = max_dicoms // 5
    frames = []
    for kl in range(5):
        stratum = pool2[pool2['kl_max'] == kl]
        n_take  = min(len(stratum), per_grade)
        if n_take > 0:
            frames.append(stratum.sample(n=n_take, random_state=42))
    plan = pd.concat(frames).reset_index(drop=True)
    print(f'Budget limit -- stratified sample: {len(plan):,} DICOMs')

print()
print('Plan by KL grade:')
for kl in range(5):
    n   = int((plan['kl_max'] == kl).sum())
    pct = 100 * n / max(len(plan), 1)
    bar = '#' * int(pct / 2)
    print(f'  KL{kl} : {n:>7,}  ({pct:.1f}%)  {bar}')

PLAN_CSV = LOG_DIR / 'download_plan.csv'
cols = ['empi_anon', 'dicom_path', 'laterality', 'view_position',
        'R_KLG_inference', 'L_KLG_inference', 'kl_max',
        'inverted', 'horizontal_flip']
plan[[c for c in cols if c in plan.columns]].to_csv(str(PLAN_CSV), index=False)

est_gb = len(plan) * avg_mb / 1024
print()
print('=' * 52)
print(f'  READY  : {len(plan):,} DICOMs')
print(f'  SIZE   : ~{est_gb:.0f} GB')
print(f'  REMAIN : ~{DRIVE_BUDGET_GB - est_gb:.0f} GB free after download')
print('=' * 52)
print('Plan saved. Run the download cell when ready.')


Measuring real DICOM sizes (100-file sample) ...
  Measured from 100 files:
  Average : 7.6 MB
  P25-P75 : 6.0 MB - 10.7 MB

Full filtered pool  : 81,104 DICOMs  (~604 GB)
Drive budget        : 1500 GB
Max DICOMs in budget: 201,295

Entire pool fits -- downloading ALL 81,104 DICOMs

Plan by KL grade:
  KL0 :  24,645  (30.4%)  ###############
  KL1 :   4,440  (5.5%)  ##
  KL2 :  29,457  (36.3%)  ##################
  KL3 :  13,923  (17.2%)  ########
  KL4 :   8,639  (10.7%)  #####

  READY  : 81,104 DICOMs
  SIZE   : ~604 GB
  REMAIN : ~896 GB free after download
Plan saved. Run the download cell when ready.


---
## 6 — Download
Parallel download with resume support.
**If Colab disconnects, just re-run this cell — completed files are skipped.**

In [9]:
import threading
import concurrent.futures
from tqdm.notebook import tqdm

PROGRESS_LOG = LOG_DIR / 'downloaded.txt'
ERROR_LOG    = LOG_DIR / 'errors.txt'
S3_IMG_PREFIX = 'images/'

if PROGRESS_LOG.exists():
    with open(PROGRESS_LOG) as f:
        done_set = set(f.read().splitlines())
    print(f'Resume: {len(done_set):,} files already done.')
else:
    done_set = set()

plan = pd.read_csv(str(PLAN_CSV))
todo = []
for _, row in plan.iterrows():
    if pd.isna(row['dicom_path']): continue
    raw_path = str(row['dicom_path'])
    s3_key   = raw_path if raw_path.startswith('images/') else S3_IMG_PREFIX + raw_path
    if s3_key not in done_set:
        todo.append((s3_key, raw_path))

print(f'Remaining : {len(todo):,}')
print(f'Done      : {len(done_set):,}')
print(f'Total plan: {len(plan):,}')
print()

if not todo:
    print('All files already downloaded.')
else:
    _lock = threading.Lock()
    ok_c = skip_c = err_c = 0

    def _dl(args):
        s3_key, raw_path = args

        parts    = raw_path.split('/')
        pat_id   = parts[0] if len(parts) > 0 else 'unknown'
        fname    = parts[-1]
        out_dir  = DICOM_DIR / pat_id
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / fname

        if out_path.exists() and out_path.stat().st_size > 0:
            with _lock:
                with open(PROGRESS_LOG, 'a') as f: f.write(s3_key + '\n')
            return 'skip', s3_key
        try:
            s3.download_file(S3_BUCKET, s3_key, str(out_path))
            with _lock:
                with open(PROGRESS_LOG, 'a') as f: f.write(s3_key + '\n')
            return 'ok', s3_key
        except Exception as e:
            if out_path.exists(): out_path.unlink()
            with _lock:
                with open(ERROR_LOG, 'a') as f: f.write(f'{s3_key}\t{e}\n')
            return 'err', s3_key

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(_dl, item): item for item in todo}
        with tqdm(total=len(todo), unit='file', desc='Downloading') as pbar:
            for fut in concurrent.futures.as_completed(futs):
                status, _ = fut.result()
                with _lock:
                    if status == 'ok':     ok_c   += 1
                    elif status == 'skip': skip_c += 1
                    else:                  err_c  += 1
                pbar.set_postfix(ok=ok_c, skip=skip_c, err=err_c, refresh=False)
                pbar.update(1)

    print()
    print(f'Downloaded : {ok_c:,}')
    print(f'Skipped    : {skip_c:,}')
    print(f'Errors     : {err_c:,}')
    if err_c:
        print(f'Error log  : {ERROR_LOG}')
        print('Re-run this cell to retry failed files.')


Remaining : 81,104
Done      : 0
Total plan: 81,104



Downloading:   0%|          | 0/81104 [00:00<?, ?file/s]


Downloaded : 81,104
Skipped    : 0
Errors     : 0


---
## 7 — Verify

In [13]:
!pip install pydicom

import random
import pydicom

all_dcm = list(DICOM_DIR.rglob('*.dcm'))
if not all_dcm:

    all_dcm = [p for p in DICOM_DIR.rglob('*') if p.is_file()]

total_gb = sum(p.stat().st_size for p in all_dcm) / 1e9

print(f'=== Download Verification ===')
print(f'  Files on Drive : {len(all_dcm):,}')
print(f'  Total size     : {total_gb:.2f} GB')
print(f'  Location       : {DICOM_DIR}')
print()

print('Spot-checking 10 random files ...')
check = random.sample(all_dcm, min(10, len(all_dcm)))
errs  = []
for p in check:
    try:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        sd = str(getattr(ds, 'StudyDescription', 'N/A'))[:40]
        print(f'  OK  {p.name:45s}  {sd}')
    except Exception as e:
        errs.append(p)
        print(f'  ERR {p.name}: {e}')

print()
if errs:
    print(f'  {len(errs)} corrupt files found. Check {ERROR_LOG}.')
else:
    print('  All spot-checks passed.')

print()
print('=== Summary ===')
plan_n = len(pd.read_csv(str(PLAN_CSV))) if PLAN_CSV.exists() else '?'
print(f'  Planned  : {plan_n}')
print(f'  On disk  : {len(all_dcm):,}')
missing = int(str(plan_n).replace(',','')) - len(all_dcm) if str(plan_n) != '?' else '?'
print(f'  Missing  : {missing}')
if isinstance(missing, int) and missing > 0:
    print('  Re-run Cell 6 to download missing files.')
elif missing == 0:
    print('  Download complete.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.2 MB/s eta 0:00:00
=== Download Verification ===
  Files on Drive : 81,104
  Total size     : 607.45 GB
  Location       : /content/drive/MyDrive/Master Thesis/MRKR/dicoms

Spot-checking 10 random files ...
  OK  1.2.826.0.1.3680043.8.498.11187401699041301118408152250067445286.dcm  XR KNEE 4 VIEWS LEFT
  OK  1.2.826.0.1.3680043.8.498.91118466741150745466875614836871551885.dcm  XR KNEE 1 OR 2 VIEWS LEFT
  OK  1.2.826.0.1.3680043.8.498.53482137968222085326747479014299499738.dcm  XR KNEE 1 OR 2 VIEWS LEFT
  OK  1.2.826.0.1.3680043.8.498.43894551360380992359515395217038020571.dcm  XR KNEE 1 OR 2 VIEWS RIGHT
  OK  1.2.826.0.1.3680043.8.498.36302116604487831078747588344458354429.dcm  XR KNEE 1 OR 2 VIEWS BILATERAL
  OK  1.2.826.0.1.3680043.8.498.10056404527397374214702469230075373943.dcm  XR KNEE 4 VIEWS RIGHT
  OK  1.2.826.0.1.3680043.8.498.14034884945742116664993632766609929629.dcm  XR KNEE 1 OR 2 VIEWS LEFT
  OK  1.2.826.0.1.368004

---
## 8 — Next Steps

DICOMs are now at `Master Thesis/MRKR/dicoms/`.

**Next notebook: `mrkr_preprocess.ipynb`**

This will:
- Load each bilateral DICOM
- Split down the midline → right crop + left crop
- Apply CLAHE (`clipLimit=2.0`) + resize to `224×224`
- Save as `MRKR_{patient_id}_{side}_clahe-2.0_224px.png`
- Build `mrkr_labels.csv` with columns:
  `filename, subject_id, side, kl_grade, split`

The output will be in exactly the same format as your OAI and NHANES3
processed data, making all three datasets interchangeable for training.